#### MCP (Model Context Protocol)
在MCP中有三个核心概念：
- MCP Server：提供MCP服务的应用，可以是远程服务，也可以是本地服务
- MCP Client：连接到MCP服务器，读取MCP信息（特别是Too/信息），供Host使用
- MCP Host：协调和管理多个MCP Client的AI应用，比如LangChain的Agent

MCP Client 与 MCP Server之间有两种通信协议：
- stdio：标准输入输出，MCP Client运行时，分两种情况：
  - 外部服务：Client会把这个MCP服务的脚本下载到本地，然后作为一个子进程运行。
  - 本地服务：Client会把本地脚本直接加载，作为一个子进程运行

  也就是说，stdio模式中，MCP Client 和 MCP Server之间的通信就是进程通信，没有网络延迟。
- streamable_http:用event stream来发送数据的http模式，本质还是Server Event Stream，也就是SSE。也就是说MCP client通过发送http请求与MCP server交互。因此存在一定的网络延迟。

详细的MCP说明可以参考Anthropic公司的MCP官方文档：
https://modelcontextprotocol.io/docs/getting-started/intro

有关通信协议可以查看：
https://modelcontextprotocol.io/specification/2025-11-25/basic/transports

时间感知：https://modelscope.cn/mcp/servers/zac2ma/Time-MCP

12306-MCP车票查询工具：https://modelscope.cn/mcp/servers/@Joooook/12306-mcp


In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langchain_mcp_adapters.client import MultiServerMCPClient
from rich import print as rich_print

"""
uv add langchain-mcp-adapters
"""

# ================ 1.加载环境变量 ================
load_dotenv()  # 读取.env文件，加入系统环境变量中

# ================ 2. 定义 MCP 客户端并加载工具 ================
client = MultiServerMCPClient(
    {
        "time-mcp": {"transport": "stdio", "args": ["-y", "time-mcp"], "command": "npx"},
        "12306-mcp": {
            "transport": "streamable_http",
            "url": "https://mcp.api-inference.modelscope.net/adce99251c7041/mcp",
        },
    }
)
# 异步获取 MCP 工具
tools = await client.get_tools()
print(f"成功加载了 {len(tools)} 个 MCP 工具：")
for item in tools:
    print(f"- {item.name}: {item.description}")

# ================ 3.定义Agent，绑定模型和工具 ================
agent = create_agent(model="deepseek-v4-flash", tools=tools)

# ================ 4.调用Agent ================
response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "帮我查一下下周四从北京到杭州高铁?"}]}
)
rich_print(response["messages"][-1].content)
rich_print(response)

##### Agent 记忆
对于智能体而言，记忆分为了两类：
- 短期记忆 （short-term memory）：当前任务或会话的上下文（Working Memory 或 Session Memory）
- 长期记忆（long-term memory）：跨任务或会话的经验与知识（Persistent Memory）

LangChain提供了自动化的记忆管理方案：
- 首先，LangChain把会话记忆（也就是Messages列表）记录在AgentState中：
- 每一次与AI的交互都会生成一个快照，记录为一个 AgentStateSnapshot，记录当前会话的状态。把同一会话的所有 AgentStateSnapshot 组合在一起，就是完整的会话历史了。
- 为了区分不同的会话记忆，不同会话需要设定各自的 thread_id，相同会话则使用相同 thread_id
- 向Agent发起会话时必须指定自己的 thread_id 以唤起对应的会话记忆

在LangChain中，所有的AgentStateSnapshot都是通过Checkpointer对象来保存的，Checkpointer 有很多不同的实现：
- InMemorySaver 基于内存存储
- SqliteSaver：基于sqlite存储
- PostgresSaver：基于Postgresql存储


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(override=True)


def run_stateless_agent():
    """无记忆配置，不传 checkpointer"""
    print("===== 演示【无记忆】（Stateless 模式）=====")
    # 【无记忆配置】不传 checkpointer
    agent = create_agent(model="deepseek-v4-flash")

    # 第一轮对话：告诉名字
    response1 = agent.invoke({"messages": [HumanMessage(content="我叫张三，记住我的名字。")]})
    rich_print("【第一轮回答】:", response1["messages"][-1].content)

    # 第二轮对话：询问名字（此时因为无状态，模型会回答不知道）
    response2 = agent.invoke({"messages": [HumanMessage(content="我叫什么名字？")]})
    rich_print("【第二轮回答（无记忆）】:", response2["messages"][-1].content)


def run_stateful_agent():
    print("===== 演示【短期记忆】（Stateful 模式）=====")
    # 1. 初始化内存检查点
    checkpointer = InMemorySaver()

    # 【短期记忆配置】传入 checkpointer
    agent = create_agent(model="deepseek-v4-flash", checkpointer=checkpointer)
    # 2. 定义会话配置（必须包含 thread_id）
    config = {"configurable": {"thread_id": "user_session_001"}}

    # 第一轮对话：告诉名字
    response1 = agent.invoke(
        {"messages": [HumanMessage(content="我叫张三，记住我的名字。")]}, config=config
    )
    rich_print("【第一轮回答】:", response1["messages"][-1].content)

    # 第二轮对话：询问名字（带上相同的 thread_id，Agent 就能记住你是张三）
    response2 = agent.invoke({"messages": [HumanMessage(content="我叫什么名字？")]}, config=config)
    rich_print("【第二轮回答（有短期记忆）】:", response2["messages"][-1].content)


run_stateless_agent()
run_stateful_agent()

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from psycopg.rows import dict_row
from psycopg_pool import AsyncConnectionPool
from rich import print as rich_print

from agent_service.core.configs.config import settings

load_dotenv(override=True)
# 创建连接池
checkpoint_pool = AsyncConnectionPool(
    conninfo=settings.db.checkpoint_url,
    min_size=1,  # 池最小连接数
    max_size=5,  # 池最大连接数
    kwargs={
        "autocommit": True,  # 自动提交事务
        "prepare_threshold": 0,  # 不做预准备sql
        "row_factory": dict_row,  # 查询结果以dict返回
    },
    open=False,  # 不自动创建连接
)
# 初始化连接
await checkpoint_pool.open()
# 等待连接池就绪
await checkpoint_pool.wait()
# 初始化checkpointer
checkpointer = AsyncPostgresSaver(checkpoint_pool)
# 自动建表（与checkpointer存储有关的表）
await checkpointer.setup()

# 使用
agent = create_agent(model="deepseek-v4-flash", checkpointer=checkpointer)
config = {"configurable": {"thread_id": "user_session_002"}}
# 第一轮对话：告诉名字
response1 = await agent.ainvoke(
    {"messages": [HumanMessage(content="我叫张三，记住我的名字。")]}, config=config
)
rich_print("【第一轮回答】:", response1["messages"][-1].content)

# 第二轮对话：询问名字（带上相同的 thread_id，Agent 就能记住你是张三）
response2 = await agent.ainvoke(
    {"messages": [HumanMessage(content="我叫什么名字？")]}, config=config
)
rich_print("【第二轮回答（有短期记忆）】:", response2["messages"][-1].content)

SSE （Server-Sent Events）是一种让服务器主动向浏览器推送数据的通信协议。
> 它基于 HTTP，使用 text/event-stream 格式，前端可以通过 EventSource API方便地接收。其核心特点就是：服务嚣一次连接，多次推送，且前端可以自动重连，非常适合像大模型输出、Agent 逐步推理这类流式场景。
>
> 简单来说，就是前端发起一次请求，服务端以stream方式不断推送新的数据给前端，前端完成渲染，从而实现大模型的流式输出效果。
>
> SSE数据格式
> 在标准的SSE中，服务端返回的数据格式是纯文本格式，可以包含event、data、id、retry字段，每个字段后跟冒号和空格，并以\n结束，不同消息之间则以两个\n结束
> ```
> [field]: [value]\n
> [field]: [value]\n
> \n\n

查询历史消息

In [17]:
agent = create_agent(model="deepseek-v4-flash", checkpointer=checkpointer)
config: RunnableConfig = {"configurable": {"thread_id": "user_session_001"}}
state = await agent.aget_state(config)
messages = state.values["messages"]
len(messages)
for message in messages:
    message.pretty_print()

================================ Human Message =================================

我叫张三，记住我的名字。
================================== Ai Message ==================================

好的，张三，我记住了。在这个对话中我会称呼你张三。
================================ Human Message =================================

我叫张三，记住我的名字。
================================== Ai Message ==================================

记住了，张三。在这个对话里我会一直记得你的名字。
================================ Human Message =================================

我叫张三，记住我的名字。
================================== Ai Message ==================================

记住了，张三。在这个对话中我会一直记得你的名字。
================================ Human Message =================================

我叫什么名字？
================================== Ai Message ==================================

你叫张三。


In [25]:
await checkpointer.adelete_thread("user_session_001")  # 删除会话历史